## 1. Import Required Libraries

# CHIRPS v3.0 TIFF File Analysis

This notebook allows you to analyze the downloaded CHIRPS v3.0 TIFF files from the `data/raw` directory. You can configure date ranges and explore the actual precipitation values.

## Features:
- Configure date ranges to analyze
- Load and inspect TIFF metadata
- Display actual precipitation values
- Visualize data with maps and plots
- Calculate statistics for selected dates

## 2. Configure Date Parameters

Set the date range you want to analyze. The notebook will find and load all TIFF files within this range.

In [1]:
import os
import sys
from pathlib import Path
from datetime import datetime, timedelta
from typing import List, Dict, Tuple

import numpy as np
import rasterio
from rasterio.plot import show
import matplotlib.pyplot as plt
import matplotlib.colors as colors
from matplotlib.patches import Rectangle
import pandas as pd

# Add parent directory to path for imports
sys.path.insert(0, str(Path.cwd().parent))

print("✓ Libraries imported successfully")

✓ Libraries imported successfully


## 3. Locate TIFF Files

Scan the `data/raw` directory and find all TIFF files matching the configured date range.

In [2]:
# ============================================================================
# CONFIGURE THESE DATES
# ============================================================================
START_DATE = "2024-01-01"  # Format: YYYY-MM-DD
END_DATE = "2024-01-31"    # Format: YYYY-MM-DD

# Optional: Analyze specific dates (leave empty to analyze all in range)
SPECIFIC_DATES = []  # Example: ["2024-01-01", "2024-01-15", "2024-02-01"]

# Display configuration
MAX_DISPLAY_FILES = 20  # Maximum number of files to show details for

# ============================================================================

# Parse dates
start_date = datetime.strptime(START_DATE, "%Y-%m-%d")
end_date = datetime.strptime(END_DATE, "%Y-%m-%d")

print(f"Configuration:")
print(f"  Start Date: {START_DATE}")
print(f"  End Date: {END_DATE}")
print(f"  Date Range: {(end_date - start_date).days + 1} days")
if SPECIFIC_DATES:
    print(f"  Specific Dates: {len(SPECIFIC_DATES)} dates selected")

Configuration:
  Start Date: 2024-01-01
  End Date: 2024-01-31
  Date Range: 31 days


## 4. Display TIFF Metadata

Examine the metadata of a sample TIFF file to understand its structure.

In [3]:
# Define paths
base_dir = Path.cwd().parent
raw_data_dir = base_dir / "data" / "raw"

def find_tiff_files(start_date: datetime, end_date: datetime, specific_dates: List[str] = None) -> Dict[str, Path]:
    """
    Find TIFF files in the raw data directory for the specified date range.
    
    Returns:
        Dictionary mapping date strings to file paths
    """
    tiff_files = {}
    
    if specific_dates:
        # Use specific dates
        dates_to_check = [datetime.strptime(d, "%Y-%m-%d") for d in specific_dates]
    else:
        # Generate date range
        dates_to_check = []
        current_date = start_date
        while current_date <= end_date:
            dates_to_check.append(current_date)
            current_date += timedelta(days=1)
    
    for date in dates_to_check:
        # CHIRPS filename pattern: chirps-v3.0.rnl.YYYY.MM.DD.tif
        year = date.year
        filename = f"chirps-v3.0.rnl.{date.year}.{date.month:02d}.{date.day:02d}.tif"
        filepath = raw_data_dir / str(year) / filename
        
        if filepath.exists():
            date_str = date.strftime("%Y-%m-%d")
            tiff_files[date_str] = filepath
    
    return tiff_files

# Find files
tiff_files = find_tiff_files(start_date, end_date, SPECIFIC_DATES)

print(f"\n{'='*80}")
print(f"Found {len(tiff_files)} TIFF files")
print(f"{'='*80}\n")

if tiff_files:
    # Display first few files
    for i, (date_str, filepath) in enumerate(list(tiff_files.items())[:MAX_DISPLAY_FILES]):
        file_size_mb = filepath.stat().st_size / (1024 * 1024)
        print(f"{i+1:3d}. {date_str}: {filepath.name} ({file_size_mb:.2f} MB)")
    
    if len(tiff_files) > MAX_DISPLAY_FILES:
        print(f"     ... and {len(tiff_files) - MAX_DISPLAY_FILES} more files")
else:
    print("⚠ No TIFF files found for the specified date range!")
    print(f"  Make sure files exist in: {raw_data_dir}")


Found 31 TIFF files

  1. 2024-01-01: chirps-v3.0.rnl.2024.01.01.tif (12.79 MB)
  2. 2024-01-02: chirps-v3.0.rnl.2024.01.02.tif (13.03 MB)
  3. 2024-01-03: chirps-v3.0.rnl.2024.01.03.tif (13.35 MB)
  4. 2024-01-04: chirps-v3.0.rnl.2024.01.04.tif (13.58 MB)
  5. 2024-01-05: chirps-v3.0.rnl.2024.01.05.tif (13.32 MB)
  6. 2024-01-06: chirps-v3.0.rnl.2024.01.06.tif (14.36 MB)
  7. 2024-01-07: chirps-v3.0.rnl.2024.01.07.tif (14.36 MB)
  8. 2024-01-08: chirps-v3.0.rnl.2024.01.08.tif (14.28 MB)
  9. 2024-01-09: chirps-v3.0.rnl.2024.01.09.tif (14.07 MB)
 10. 2024-01-10: chirps-v3.0.rnl.2024.01.10.tif (14.04 MB)
 11. 2024-01-11: chirps-v3.0.rnl.2024.01.11.tif (13.92 MB)
 12. 2024-01-12: chirps-v3.0.rnl.2024.01.12.tif (14.26 MB)
 13. 2024-01-13: chirps-v3.0.rnl.2024.01.13.tif (14.09 MB)
 14. 2024-01-14: chirps-v3.0.rnl.2024.01.14.tif (14.12 MB)
 15. 2024-01-15: chirps-v3.0.rnl.2024.01.15.tif (14.10 MB)
 16. 2024-01-16: chirps-v3.0.rnl.2024.01.16.tif (12.62 MB)
 17. 2024-01-17: chirps-v3.0.rnl.2

## 5. Read and Display Actual Precipitation Values

Load the raster data and display actual pixel values from the TIFF files.

In [4]:
if tiff_files:
    # Select a file to display values
    sample_date = list(tiff_files.keys())[0]
    sample_file = tiff_files[sample_date]
    
    print(f"\n{'='*80}")
    print(f"Precipitation Values for: {sample_date}")
    print(f"{'='*80}\n")
    
    with rasterio.open(sample_file) as src:
        # Read the data
        data = src.read(1)  # Read first (and only) band
        
        print(f"Array Shape: {data.shape}")
        print(f"Data Type: {data.dtype}")
        
        # Filter out NoData values
        valid_data = data[data != -9999.0]
        nodata_count = np.sum(data == -9999.0)
        valid_count = data.size - nodata_count
        
        print(f"\nData Coverage:")
        print(f"  Total pixels:  {data.size:,}")
        print(f"  Valid pixels:  {valid_count:,} ({100 * valid_count / data.size:.1f}%)")
        print(f"  NoData pixels: {nodata_count:,} ({100 * nodata_count / data.size:.1f}%)")
        
        if len(valid_data) > 0:
            print(f"\nPrecipitation Statistics (mm/day):")
            print(f"  Min:    {np.min(valid_data):.4f}")
            print(f"  Max:    {np.max(valid_data):.4f}")
            print(f"  Mean:   {np.mean(valid_data):.4f}")
            print(f"  Median: {np.median(valid_data):.4f}")
            print(f"  Std:    {np.std(valid_data):.4f}")
            
            # Display sample values from different regions
            print(f"\nSample Values (first 10x10 pixels, top-left corner):")
            sample_region = data[:10, :10]
            print(sample_region)
            
            # Show value distribution
            print(f"\nValue Distribution:")
            percentiles = [0, 10, 25, 50, 75, 90, 95, 99, 100]
            for p in percentiles:
                val = np.percentile(valid_data, p)
                print(f"  {p:3d}th percentile: {val:.4f} mm/day")
        else:
            print("\n⚠ No valid data found (all pixels are NoData)")
else:
    print("No files to analyze.")


Precipitation Values for: 2024-01-01

Array Shape: (2400, 7200)
Data Type: float32

Data Coverage:
  Total pixels:  17,280,000
  Valid pixels:  4,841,966 (28.0%)
  NoData pixels: 12,438,034 (72.0%)

Precipitation Statistics (mm/day):
  Min:    0.0000
  Max:    162.2346
  Mean:   2.1606
  Median: 0.0000
  Std:    5.7289

Sample Values (first 10x10 pixels, top-left corner):
[[-9999. -9999. -9999. -9999. -9999. -9999. -9999. -9999. -9999. -9999.]
 [-9999. -9999. -9999. -9999. -9999. -9999. -9999. -9999. -9999. -9999.]
 [-9999. -9999. -9999. -9999. -9999. -9999. -9999. -9999. -9999. -9999.]
 [-9999. -9999. -9999. -9999. -9999. -9999. -9999. -9999. -9999. -9999.]
 [-9999. -9999. -9999. -9999. -9999. -9999. -9999. -9999. -9999. -9999.]
 [-9999. -9999. -9999. -9999. -9999. -9999. -9999. -9999. -9999. -9999.]
 [-9999. -9999. -9999. -9999. -9999. -9999. -9999. -9999. -9999. -9999.]
 [-9999. -9999. -9999. -9999. -9999. -9999. -9999. -9999. -9999. -9999.]
 [-9999. -9999. -9999. -9999. -9999. -99

## 6. Visualize TIFF Data

Create visualizations of the precipitation data.

In [ ]:
if tiff_files:
    # Select file to visualize
    viz_date = list(tiff_files.keys())[0]
    viz_file = tiff_files[viz_date]
    
    with rasterio.open(viz_file) as src:
        data = src.read(1)
        
        # Mask NoData values for visualization
        masked_data = np.ma.masked_where(data == -9999.0, data)
        
        # Create figure with subplots
        fig, axes = plt.subplots(2, 2, figsize=(16, 12))
        fig.suptitle(f'CHIRPS Precipitation Analysis - {viz_date}', fontsize=16, fontweight='bold')
        
        # 1. Full global map
        ax1 = axes[0, 0]
        im1 = ax1.imshow(masked_data, cmap='Blues', interpolation='nearest')
        ax1.set_title('Global Precipitation Distribution')
        ax1.set_xlabel('Longitude (pixel)')
        ax1.set_ylabel('Latitude (pixel)')
        plt.colorbar(im1, ax=ax1, label='Precipitation (mm/day)', shrink=0.8)
        
        # 2. Logarithmic scale (for better visualization of range)
        ax2 = axes[0, 1]
        # Add small value to avoid log(0)
        log_data = np.ma.masked_where(data == -9999.0, data + 0.01)
        im2 = ax2.imshow(log_data, cmap='YlGnBu', norm=colors.LogNorm(vmin=0.01, vmax=log_data.max()), 
                        interpolation='nearest')
        ax2.set_title('Precipitation (Logarithmic Scale)')
        ax2.set_xlabel('Longitude (pixel)')
        ax2.set_ylabel('Latitude (pixel)')
        plt.colorbar(im2, ax=ax2, label='Precipitation (mm/day, log scale)', shrink=0.8)
        
        # 3. Histogram
        ax3 = axes[1, 0]
        valid_data = data[data != -9999.0]
        if len(valid_data) > 0:
            ax3.hist(valid_data, bins=50, edgecolor='black', alpha=0.7, color='steelblue')
            ax3.set_xlabel('Precipitation (mm/day)')
            ax3.set_ylabel('Frequency')
            ax3.set_title('Precipitation Value Distribution')
            ax3.axvline(np.mean(valid_data), color='red', linestyle='--', linewidth=2, label=f'Mean: {np.mean(valid_data):.2f}')
            ax3.axvline(np.median(valid_data), color='green', linestyle='--', linewidth=2, label=f'Median: {np.median(valid_data):.2f}')
            ax3.legend()
            ax3.grid(True, alpha=0.3)
        
        # 4. Box plot by latitude bands
        ax4 = axes[1, 1]
        # Divide into latitude bands
        n_bands = 10
        band_height = data.shape[0] // n_bands
        band_data = []
        band_labels = []
        
        for i in range(n_bands):
            start_row = i * band_height
            end_row = (i + 1) * band_height if i < n_bands - 1 else data.shape[0]
            band = data[start_row:end_row, :]
            valid_band = band[band != -9999.0]
            if len(valid_band) > 0:
                band_data.append(valid_band)
                # Approximate latitude (assuming global coverage -50 to 50)
                lat_start = 50 - (i * 100 / n_bands)
                lat_end = 50 - ((i + 1) * 100 / n_bands)
                band_labels.append(f'{lat_end:.0f}° to {lat_start:.0f}°')
        
        if band_data:
            bp = ax4.boxplot(band_data, labels=band_labels, patch_artist=True)
            for patch in bp['boxes']:
                patch.set_facecolor('lightblue')
            ax4.set_xlabel('Latitude Band')
            ax4.set_ylabel('Precipitation (mm/day)')
            ax4.set_title('Precipitation Distribution by Latitude')
            ax4.grid(True, alpha=0.3, axis='y')
            plt.setp(ax4.xaxis.get_majorticklabels(), rotation=45, ha='right')
        
        plt.tight_layout()
        plt.show()
        
        print(f"✓ Visualization complete for {viz_date}")
else:
    print("No files to visualize.")

## 7. Regional Zoom View

Zoom into a specific region to see more detail.

In [ ]:
if tiff_files:
    # Define region to zoom (lon_min, lon_max, lat_min, lat_max)
    # Example: East Africa
    region = {
        'name': 'East Africa',
        'lon_min': 25.0,
        'lon_max': 50.0,
        'lat_min': -15.0,
        'lat_max': 15.0
    }
    
    viz_date = list(tiff_files.keys())[0]
    viz_file = tiff_files[viz_date]
    
    with rasterio.open(viz_file) as src:
        # Convert geographic coordinates to pixel coordinates
        # CHIRPS is global: -180 to 180 longitude, -50 to 50 latitude at 0.05° resolution
        lon_min_global = -180.0
        lat_max_global = 50.0
        resolution = 0.05
        
        # Calculate pixel indices
        col_min = int((region['lon_min'] - lon_min_global) / resolution)
        col_max = int((region['lon_max'] - lon_min_global) / resolution)
        row_min = int((lat_max_global - region['lat_max']) / resolution)
        row_max = int((lat_max_global - region['lat_min']) / resolution)
        
        # Read the subset
        data = src.read(1)
        subset = data[row_min:row_max, col_min:col_max]
        
        # Mask NoData
        masked_subset = np.ma.masked_where(subset == -9999.0, subset)
        
        # Create visualization
        fig, axes = plt.subplots(1, 2, figsize=(16, 6))
        fig.suptitle(f'{region["name"]} - Precipitation on {viz_date}', fontsize=14, fontweight='bold')
        
        # Regional map
        ax1 = axes[0]
        im1 = ax1.imshow(masked_subset, cmap='Blues', interpolation='nearest', aspect='auto')
        ax1.set_title(f'{region["name"]} Detail')
        ax1.set_xlabel('Longitude (pixel)')
        ax1.set_ylabel('Latitude (pixel)')
        plt.colorbar(im1, ax=ax1, label='Precipitation (mm/day)', shrink=0.8)
        
        # Add coordinate labels
        ax1.set_xticks([0, masked_subset.shape[1]//2, masked_subset.shape[1]-1])
        ax1.set_xticklabels([f"{region['lon_min']:.1f}°", 
                             f"{(region['lon_min']+region['lon_max'])/2:.1f}°",
                             f"{region['lon_max']:.1f}°"])
        ax1.set_yticks([0, masked_subset.shape[0]//2, masked_subset.shape[0]-1])
        ax1.set_yticklabels([f"{region['lat_max']:.1f}°",
                             f"{(region['lat_min']+region['lat_max'])/2:.1f}°",
                             f"{region['lat_min']:.1f}°"])
        
        # Statistics
        ax2 = axes[1]
        ax2.axis('off')
        
        valid_subset = subset[subset != -9999.0]
        if len(valid_subset) > 0:
            stats_text = f"""
Region: {region['name']}
Date: {viz_date}

Spatial Extent:
  Longitude: {region['lon_min']}° to {region['lon_max']}°
  Latitude: {region['lat_min']}° to {region['lat_max']}°

Data Dimensions:
  Rows: {subset.shape[0]}
  Columns: {subset.shape[1]}
  Total Pixels: {subset.size:,}

Data Coverage:
  Valid Pixels: {len(valid_subset):,} ({100*len(valid_subset)/subset.size:.1f}%)
  NoData Pixels: {subset.size - len(valid_subset):,}

Precipitation Statistics (mm/day):
  Minimum:  {np.min(valid_subset):.4f}
  Maximum:  {np.max(valid_subset):.4f}
  Mean:     {np.mean(valid_subset):.4f}
  Median:   {np.median(valid_subset):.4f}
  Std Dev:  {np.std(valid_subset):.4f}
  
Percentiles:
  25th: {np.percentile(valid_subset, 25):.4f}
  75th: {np.percentile(valid_subset, 75):.4f}
  90th: {np.percentile(valid_subset, 90):.4f}
  95th: {np.percentile(valid_subset, 95):.4f}
            """
            ax2.text(0.1, 0.95, stats_text, transform=ax2.transAxes,
                    fontsize=11, verticalalignment='top', fontfamily='monospace',
                    bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.3))
        
        plt.tight_layout()
        plt.show()
        
        print(f"✓ Regional visualization complete")
else:
    print("No files to visualize.")

## 8. Multi-Date Comparison

Compare precipitation across multiple dates in the selected range.

In [ ]:
if tiff_files and len(tiff_files) >= 2:
    # Limit to first N dates for comparison
    max_compare = min(4, len(tiff_files))
    dates_to_compare = list(tiff_files.keys())[:max_compare]
    
    print(f"Comparing {len(dates_to_compare)} dates: {', '.join(dates_to_compare)}\n")
    
    # Create subplot grid
    fig, axes = plt.subplots(2, max_compare, figsize=(5*max_compare, 10))
    if max_compare == 1:
        axes = axes.reshape(2, 1)
    
    fig.suptitle(f'Multi-Date Precipitation Comparison', fontsize=16, fontweight='bold')
    
    stats_data = []
    
    for i, date in enumerate(dates_to_compare):
        filepath = tiff_files[date]
        
        with rasterio.open(filepath) as src:
            data = src.read(1)
            masked_data = np.ma.masked_where(data == -9999.0, data)
            valid_data = data[data != -9999.0]
            
            # Top row: Maps
            ax_map = axes[0, i]
            im = ax_map.imshow(masked_data, cmap='Blues', interpolation='nearest')
            ax_map.set_title(f'{date}', fontweight='bold')
            ax_map.set_xlabel('Longitude (pixel)')
            if i == 0:
                ax_map.set_ylabel('Latitude (pixel)')
            plt.colorbar(im, ax=ax_map, label='mm/day', shrink=0.7)
            
            # Bottom row: Histograms
            ax_hist = axes[1, i]
            if len(valid_data) > 0:
                ax_hist.hist(valid_data, bins=30, edgecolor='black', alpha=0.7, color='steelblue')
                ax_hist.set_xlabel('Precipitation (mm/day)')
                if i == 0:
                    ax_hist.set_ylabel('Frequency')
                ax_hist.set_title(f'Mean: {np.mean(valid_data):.2f} mm/day')
                ax_hist.grid(True, alpha=0.3)
                
                # Collect statistics
                stats_data.append({
                    'Date': date,
                    'Mean': np.mean(valid_data),
                    'Median': np.median(valid_data),
                    'Std': np.std(valid_data),
                    'Min': np.min(valid_data),
                    'Max': np.max(valid_data),
                    'Valid %': 100 * len(valid_data) / data.size
                })
    
    plt.tight_layout()
    plt.show()
    
    # Display statistics table
    if stats_data:
        print("\n" + "="*80)
        print("STATISTICS SUMMARY")
        print("="*80 + "\n")
        
        df = pd.DataFrame(stats_data)
        print(df.to_string(index=False))
        
elif tiff_files and len(tiff_files) == 1:
    print("Only one date available. Need at least 2 dates for comparison.")
else:
    print("No files to compare.")

## 9. Time Series Analysis

Calculate statistics across all dates in the selected range.

In [ ]:
if tiff_files:
    print(f"Analyzing time series for {len(tiff_files)} dates...\n")
    
    # Collect time series data
    time_series_data = []
    
    for date_str, filepath in tiff_files.items():
        with rasterio.open(filepath) as src:
            data = src.read(1)
            valid_data = data[data != -9999.0]
            
            if len(valid_data) > 0:
                time_series_data.append({
                    'date': datetime.strptime(date_str, "%Y-%m-%d"),
                    'mean': np.mean(valid_data),
                    'median': np.median(valid_data),
                    'min': np.min(valid_data),
                    'max': np.max(valid_data),
                    'std': np.std(valid_data),
                    'valid_percent': 100 * len(valid_data) / data.size
                })
    
    if time_series_data:
        # Create DataFrame
        df = pd.DataFrame(time_series_data)
        df = df.sort_values('date')
        
        # Create visualization
        fig, axes = plt.subplots(3, 1, figsize=(14, 12))
        fig.suptitle(f'Time Series Analysis: {START_DATE} to {END_DATE}', fontsize=16, fontweight='bold')
        
        # Plot 1: Mean and Median
        ax1 = axes[0]
        ax1.plot(df['date'], df['mean'], marker='o', linewidth=2, label='Mean', color='steelblue')
        ax1.plot(df['date'], df['median'], marker='s', linewidth=2, label='Median', color='coral', alpha=0.7)
        ax1.fill_between(df['date'], df['mean'] - df['std'], df['mean'] + df['std'], alpha=0.2, color='steelblue')
        ax1.set_ylabel('Precipitation (mm/day)', fontsize=12)
        ax1.set_title('Mean and Median Precipitation Over Time', fontweight='bold')
        ax1.legend()
        ax1.grid(True, alpha=0.3)
        
        # Plot 2: Min and Max Range
        ax2 = axes[1]
        ax2.fill_between(df['date'], df['min'], df['max'], alpha=0.3, color='green', label='Min-Max Range')
        ax2.plot(df['date'], df['max'], marker='^', linewidth=1.5, label='Maximum', color='darkgreen')
        ax2.plot(df['date'], df['min'], marker='v', linewidth=1.5, label='Minimum', color='olive')
        ax2.set_ylabel('Precipitation (mm/day)', fontsize=12)
        ax2.set_title('Precipitation Range (Min-Max)', fontweight='bold')
        ax2.legend()
        ax2.grid(True, alpha=0.3)
        
        # Plot 3: Valid Data Percentage
        ax3 = axes[2]
        ax3.bar(df['date'], df['valid_percent'], color='purple', alpha=0.6, edgecolor='black')
        ax3.set_ylabel('Valid Data (%)', fontsize=12)
        ax3.set_xlabel('Date', fontsize=12)
        ax3.set_title('Data Coverage (% Valid Pixels)', fontweight='bold')
        ax3.axhline(y=50, color='red', linestyle='--', linewidth=1, label='50% threshold')
        ax3.legend()
        ax3.grid(True, alpha=0.3, axis='y')
        
        # Format x-axis
        for ax in axes:
            ax.tick_params(axis='x', rotation=45)
        
        plt.tight_layout()
        plt.show()
        
        # Display summary statistics
        print("\n" + "="*80)
        print("TIME SERIES SUMMARY STATISTICS")
        print("="*80 + "\n")
        
        print(f"Period: {df['date'].min().strftime('%Y-%m-%d')} to {df['date'].max().strftime('%Y-%m-%d')}")
        print(f"Number of Days: {len(df)}\n")
        
        print("Overall Statistics (mm/day):")
        print(f"  Mean of Daily Means:   {df['mean'].mean():.4f}")
        print(f"  Std of Daily Means:    {df['mean'].std():.4f}")
        print(f"  Minimum Daily Mean:    {df['mean'].min():.4f} (on {df.loc[df['mean'].idxmin(), 'date'].strftime('%Y-%m-%d')})")
        print(f"  Maximum Daily Mean:    {df['mean'].max():.4f} (on {df.loc[df['mean'].idxmax(), 'date'].strftime('%Y-%m-%d')})")
        print(f"  Global Maximum Value:  {df['max'].max():.4f}")
        print(f"  Average Data Coverage: {df['valid_percent'].mean():.1f}%")
        
    else:
        print("No valid data found in any of the files.")
        
else:
    print("No files to analyze.")

## 10. Export Sample Data

Export a sample of the data for further analysis.

In [ ]:
if tiff_files:
    # Create a summary DataFrame
    summary_data = []
    
    print(f"Extracting summary data from {len(tiff_files)} files...\n")
    
    for date_str, filepath in list(tiff_files.items())[:10]:  # Limit to first 10 for quick processing
        with rasterio.open(filepath) as src:
            data = src.read(1)
            valid_data = data[data != -9999.0]
            
            if len(valid_data) > 0:
                summary_data.append({
                    'Date': date_str,
                    'File': filepath.name,
                    'Total_Pixels': data.size,
                    'Valid_Pixels': len(valid_data),
                    'Valid_Percent': 100 * len(valid_data) / data.size,
                    'Mean_Precip_mm': np.mean(valid_data),
                    'Median_Precip_mm': np.median(valid_data),
                    'Min_Precip_mm': np.min(valid_data),
                    'Max_Precip_mm': np.max(valid_data),
                    'Std_Precip_mm': np.std(valid_data),
                    'P25_mm': np.percentile(valid_data, 25),
                    'P75_mm': np.percentile(valid_data, 75),
                    'P95_mm': np.percentile(valid_data, 95),
                })
    
    if summary_data:
        df_summary = pd.DataFrame(summary_data)
        
        print("="*80)
        print("SUMMARY DATA TABLE")
        print("="*80 + "\n")
        print(df_summary.to_string(index=False))
        
        # Optionally save to CSV
        output_file = base_dir / "notebooks" / f"chirps_summary_{START_DATE}_to_{END_DATE}.csv"
        df_summary.to_csv(output_file, index=False)
        print(f"\n✓ Summary data saved to: {output_file}")
    else:
        print("No data to export.")
else:
    print("No files to export.")